In [ ]:
pip install torch

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import numpy as np
import cv2
import matplotlib.pyplot as plt
from skimage import color
import os
from ultralytics import YOLO
import albumentations as A
from albumentations.pytorch import ToTensorV2

class ImageColorizationDataset(Dataset):
    """Custom dataset for image colorization"""
    
    def __init__(self, image_paths, transform=None, target_size=(256, 256)):
        self.image_paths = image_paths
        self.transform = transform
        self.target_size = target_size
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load color image
        color_img = cv2.imread(self.image_paths[idx])
        color_img = cv2.cvtColor(color_img, cv2.COLOR_BGR2RGB)
        color_img = cv2.resize(color_img, self.target_size)
        
        # Convert to LAB color space
        lab_img = color.rgb2lab(color_img / 255.0)
        
        # Split channels
        L_channel = lab_img[:, :, 0]  # Lightness (grayscale)
        ab_channels = lab_img[:, :, 1:]  # a and b channels (color)
        
        # Normalize
        L_channel = (L_channel / 50.0) - 1.0  # Normalize L to [-1, 1]
        ab_channels = ab_channels / 110.0  # Normalize ab to [-1, 1]
        
        # Convert to tensors
        L_tensor = torch.FloatTensor(L_channel).unsqueeze(0)
        ab_tensor = torch.FloatTensor(ab_channels.transpose(2, 0, 1))
        
        return L_tensor, ab_tensor

class ResidualBlock(nn.Module):
    """Residual block for better gradient flow"""
    
    def __init__(self, in_channels, out_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class AttentionBlock(nn.Module):
    """Self-attention mechanism for better feature representation"""
    
    def __init__(self, in_channels):
        super(AttentionBlock, self).__init__()
        self.query = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.key = nn.Conv2d(in_channels, in_channels // 8, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.softmax = nn.Softmax(dim=-1)
        
    def forward(self, x):
        batch_size, channels, height, width = x.size()
        
        # Generate query, key, value
        query = self.query(x).view(batch_size, -1, width * height).permute(0, 2, 1)
        key = self.key(x).view(batch_size, -1, width * height)
        value = self.value(x).view(batch_size, -1, width * height)
        
        # Calculate attention
        attention = torch.bmm(query, key)
        attention = self.softmax(attention)
        
        # Apply attention to value
        out = torch.bmm(value, attention.permute(0, 2, 1))
        out = out.view(batch_size, channels, height, width)
        
        return self.gamma * out + x

class AdvancedColorizer(nn.Module):
    """Advanced colorization model with ResNet backbone and attention mechanism"""
    
    def __init__(self):
        super(AdvancedColorizer, self).__init__()
        
        # Encoder (based on ResNet-18 but modified for single channel input)
        resnet = models.resnet18(pretrained=True)
        
        # Modify first layer for single channel (L channel) input
        self.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.conv1.weight.data = resnet.conv1.weight.data.mean(dim=1, keepdim=True)
        
        self.bn1 = resnet.bn1
        self.relu = resnet.relu
        self.maxpool = resnet.maxpool
        
        # Encoder layers
        self.layer1 = resnet.layer1  # 64 -> 64
        self.layer2 = resnet.layer2  # 64 -> 128
        self.layer3 = resnet.layer3  # 128 -> 256
        self.layer4 = resnet.layer4  # 256 -> 512
        
        # Attention mechanism
        self.attention = AttentionBlock(512)
        
        # Decoder with skip connections
        self.decoder4 = self._make_decoder_block(512, 256)
        self.decoder3 = self._make_decoder_block(256 + 256, 128)  # +256 for skip connection
        self.decoder2 = self._make_decoder_block(128 + 128, 64)   # +128 for skip connection
        self.decoder1 = self._make_decoder_block(64 + 64, 32)     # +64 for skip connection
        
        # Final layers
        self.final_conv = nn.Sequential(
            nn.Conv2d(32, 16, 3, 1, 1),
            nn.ReLU(),
            nn.Conv2d(16, 2, 3, 1, 1),  # Output 2 channels (a, b)
            nn.Tanh()  # Output range [-1, 1]
        )
        
        # Upsampling for final output
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        
    def _make_decoder_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            ResidualBlock(out_channels, out_channels)
        )
    
    def forward(self, x):
        # Encoder
        x1 = self.relu(self.bn1(self.conv1(x)))
        x1_pool = self.maxpool(x1)
        
        x2 = self.layer1(x1_pool)  # 64 channels
        x3 = self.layer2(x2)       # 128 channels
        x4 = self.layer3(x3)       # 256 channels
        x5 = self.layer4(x4)       # 512 channels
        
        # Apply attention
        x5_att = self.attention(x5)
        
        # Decoder with skip connections
        d4 = self.decoder4(x5_att)
        d3 = self.decoder3(torch.cat([d4, x4], dim=1))
        d2 = self.decoder2(torch.cat([d3, x3], dim=1))
        d1 = self.decoder1(torch.cat([d2, x2], dim=1))
        
        # Final output
        out = self.final_conv(d1)
        out = self.upsample(out)  # Ensure output size matches input
        
        return out

class YOLOEnhancedColorizer:
    """Enhanced colorizer using YOLO for object detection and context-aware colorization"""
    
    def __init__(self, colorizer_model, yolo_model_path='yolov8n.pt'):
        self.colorizer = colorizer_model
        self.yolo = YOLO(yolo_model_path)
        
        # Object-specific color palettes (simplified)
        self.object_colors = {
            'person': {'skin': [0.2, 0.1], 'clothing': [0.0, 0.0]},
            'car': {'body': [-0.1, 0.2], 'tire': [0.0, 0.0]},
            'tree': {'leaves': [-0.3, 0.2], 'trunk': [0.1, 0.3]},
            'sky': {'clear': [0.1, -0.4], 'cloudy': [0.0, -0.2]},
            'grass': {'green': [-0.4, 0.3]},
            'building': {'concrete': [0.0, 0.1], 'brick': [0.2, 0.3]}
        }
    
    def detect_objects(self, image):
        """Detect objects in the image using YOLO"""
        results = self.yolo(image)
        objects = []
        
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                for box in boxes:
                    x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
                    confidence = box.conf[0].cpu().numpy()
                    class_id = int(box.cls[0].cpu().numpy())
                    class_name = self.yolo.names[class_id]
                    
                    objects.append({
                        'bbox': [int(x1), int(y1), int(x2), int(y2)],
                        'class': class_name,
                        'confidence': confidence
                    })
        
        return objects
    
    def enhance_colorization(self, L_channel, predicted_ab, objects):
        """Enhance colorization based on detected objects"""
        enhanced_ab = predicted_ab.clone()
        
        for obj in objects:
            if obj['confidence'] > 0.5 and obj['class'] in self.object_colors:
                x1, y1, x2, y2 = obj['bbox']
                
                # Get object-specific colors
                obj_colors = self.object_colors[obj['class']]
                
                # Apply color enhancement to the detected region
                for color_type, ab_values in obj_colors.items():
                    # Simple enhancement - can be made more sophisticated
                    enhanced_ab[0, :, y1:y2, x1:x2] = (
                        enhanced_ab[0, :, y1:y2, x1:x2] * 0.7 + 
                        torch.tensor(ab_values).view(2, 1, 1) * 0.3
                    )
        
        return enhanced_ab

class ColorizationTrainer:
    """Training class for the colorization model"""
    
    def __init__(self, model, device='cuda' if torch.cuda.is_available() else 'cpu'):
        self.model = model.to(device)
        self.device = device
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=0.001, weight_decay=1e-4)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, patience=5)
        
    def train_epoch(self, dataloader):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        
        for batch_idx, (L_batch, ab_batch) in enumerate(dataloader):
            L_batch, ab_batch = L_batch.to(self.device), ab_batch.to(self.device)
            
            self.optimizer.zero_grad()
            
            # Forward pass
            predicted_ab = self.model(L_batch)
            loss = self.criterion(predicted_ab, ab_batch)
            
            # Backward pass
            loss.backward()
            self.optimizer.step()
            
            total_loss += loss.item()
            
            if batch_idx % 100 == 0:
                print(f'Batch {batch_idx}, Loss: {loss.item():.4f}')
        
        return total_loss / len(dataloader)
    
    def validate(self, dataloader):
        """Validate the model"""
        self.model.eval()
        total_loss = 0
        
        with torch.no_grad():
            for L_batch, ab_batch in dataloader:
                L_batch, ab_batch = L_batch.to(self.device), ab_batch.to(self.device)
                predicted_ab = self.model(L_batch)
                loss = self.criterion(predicted_ab, ab_batch)
                total_loss += loss.item()
        
        return total_loss / len(dataloader)
    
    def train(self, train_loader, val_loader, epochs=50):
        """Full training loop"""
        best_val_loss = float('inf')
        
        for epoch in range(epochs):
            print(f'Epoch {epoch+1}/{epochs}')
            
            # Train
            train_loss = self.train_epoch(train_loader)
            
            # Validate
            val_loss = self.validate(val_loader)
            
            # Update learning rate
            self.scheduler.step(val_loss)
            
            print(f'Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')
            
            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(self.model.state_dict(), 'best_colorizer.pth')
                print('Saved best model')
            
            print('-' * 50)

def lab_to_rgb(L, ab):
    """Convert LAB to RGB"""
    L = (L + 1.0) * 50.0
    ab = ab * 110.0
    
    Lab = torch.cat([L, ab], dim=1).permute(0, 2, 3, 1).cpu().numpy()
    rgb_imgs = []
    
    for lab_img in Lab:
        rgb_img = color.lab2rgb(lab_img)
        rgb_imgs.append(rgb_img)
    
    return np.array(rgb_imgs)

def colorize_image(model, image_path, yolo_enhanced=True):
    """Colorize a single image"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    
    # Load and preprocess image
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256, 256))
    
    # Convert to LAB
    lab_img = color.rgb2lab(img / 255.0)
    L_channel = lab_img[:, :, 0]
    L_channel = (L_channel / 50.0) - 1.0
    
    # Convert to tensor
    L_tensor = torch.FloatTensor(L_channel).unsqueeze(0).unsqueeze(0).to(device)
    
    with torch.no_grad():
        predicted_ab = model(L_tensor)
        
        if yolo_enhanced:
            # Use YOLO enhancement
            yolo_colorizer = YOLOEnhancedColorizer(model)
            objects = yolo_colorizer.detect_objects(img)
            predicted_ab = yolo_colorizer.enhance_colorization(L_tensor, predicted_ab, objects)
    
    # Convert back to RGB
    colorized = lab_to_rgb(L_tensor.cpu(), predicted_ab.cpu())
    
    return colorized[0], img

def main():
    """Main function to demonstrate usage"""
    
    # Initialize model
    model = AdvancedColorizer()
    
    # Example usage for training
    print("Advanced Image Colorization Model Initialized!")
    print("Features:")
    print("- ResNet-18 encoder with attention mechanism")
    print("- Skip connections in decoder")
    print("- YOLO integration for object-aware colorization")
    print("- LAB color space processing")
    
    # Example training setup (uncomment to use)
    """
    # Prepare dataset
    image_paths = ['path/to/image1.jpg', 'path/to/image2.jpg']  # Add your image paths
    dataset = ImageColorizationDataset(image_paths)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)
    
    # Split into train/val
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
    
    # Train model
    trainer = ColorizationTrainer(model)
    trainer.train(train_loader, val_loader, epochs=50)
    """
    
    print("\nTo use this model:")
    print("1. Prepare your dataset of color images")
    print("2. Initialize the training setup")
    print("3. Train the model using the ColorizationTrainer")
    print("4. Use colorize_image() function to colorize new images")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'torch'